# DeepSense-X — Phase 1: MetroPT-3 Data Understanding & Preparation

**Project:** Explainable and Uncertainty-Aware Hybrid LSTM–Transformer Framework for Industrial Anomaly Detection and Predictive Maintenance

**Dataset:** MetroPT-3 (UCI Machine Learning Repository)  
**Primary Phase 1 objective:** build a leakage-safe, chronological data pipeline for compressor anomaly detection and carefully qualified early-failure-warning experiments.

> **Important research note:** MetroPT-3 contains operational sensor readings and a small number of documented air-leak failure periods. It does **not** provide continuous run-to-failure Remaining Useful Life (RUL) labels. This notebook therefore prepares the data for **anomaly detection** and an exploratory **future failure-warning** task. Do not report this as a conventional supervised RUL-regression dataset.

### What this notebook does
1. Installs/imports libraries and configures Google Drive.
2. Downloads MetroPT-3 through the official UCI dataset interface.
3. Inspects the real CSV schema instead of assuming a fixed number of columns.
4. Audits timestamps, sampling intervals, duplicates, missing values and sensor types.
5. Visualizes sensor behaviour and documents the reported failure windows.
6. Creates chronological train/validation/test partitions.
7. Fits preprocessing only on the training period.
8. Builds sequence windows without crossing time splits or large timestamp gaps.
9. Creates anomaly and future-failure labels with explicit caveats.
10. Saves reusable processed arrays and metadata for Phase 2 model training.

**Recommended environment:** Google Colab. The raw dataset is large; use Colab/Drive rather than repeatedly processing it on an 8 GB laptop.

## 0. Dataset and research references

- UCI dataset page: https://archive.ics.uci.edu/dataset/791/metropt+3+
- Dataset paper: Veloso et al., *A Dataset for Predictive Maintenance of Air Pressure Systems in Metro Trains*, Scientific Data (2022): https://www.nature.com/articles/s41597-022-01877-3
- Dataset DOI: `10.24432/C5VW3R`
- License: CC BY 4.0. Cite the dataset authors and UCI repository in your report and acknowledge the dataset license.

The UCI documentation recommends using an early period for training and later periods for testing. Here we use **February 2020 for training**, **March 2020 for validation**, and **April–August 2020 for chronological testing** as an initial experimental protocol. Confirm the actual date coverage after loading the file.

## 1. Install and import libraries

In [ ]:
!pip -q install ucimlrepo pyarrow joblib

In [ ]:
import os
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    average_precision_score,
)

import joblib
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

## 2. Mount Google Drive and create project folders

If you are not using Colab, comment out the Drive mount and set `PROJECT_DIR` to a local folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/DeepSense_X_MetroPT3')
RAW_DIR = PROJECT_DIR / 'raw'
PROCESSED_DIR = PROJECT_DIR / 'processed'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'

for folder in [RAW_DIR, PROCESSED_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

## 3. Download MetroPT-3 from UCI

The UCI Python package is the preferred route. If the download fails, retry later or download the dataset from the official UCI page and upload the CSV into `RAW_DIR`. The code below keeps the feature table as a DataFrame and inspects its columns.

In [ ]:
metropt = fetch_ucirepo(id=791)

# UCI package exposes the data as a pandas DataFrame.
df_raw = metropt.data.features.copy()

# Some dataset releases may expose the target separately. Preserve it if present,
# but this project does not rely on a pre-existing RUL target.
if metropt.data.targets is not None:
    targets = metropt.data.targets.copy()
    print("Separate target table shape:", targets.shape)
else:
    targets = None

print("Raw feature table shape:", df_raw.shape)
display(df_raw.head())
print("\\nColumns returned by UCI:")
print(df_raw.columns.tolist())

### If the UCI package download fails

Use the official UCI dataset page linked above. Download the dataset archive, extract it, and upload the CSV file to `RAW_DIR`. Then replace the download cell's `df_raw = ...` line with the following (edit the filename if needed):

```python
csv_files = list(RAW_DIR.glob('*.csv'))
print(csv_files)
df_raw = pd.read_csv(csv_files[0])
```

Do not use an unofficial mirror unless you have verified that it is the same dataset version.

## 4. Inspect schema and identify timestamp/index columns

Do not hard-code the number of columns. UCI metadata summaries and file versions may present counts differently; the CSV header is the authority for this run.

In [ ]:
print("Shape:", df_raw.shape)
print("\\nDtypes:")
display(df_raw.dtypes.to_frame("dtype"))

print("\\nColumn names:")
for i, col in enumerate(df_raw.columns):
    print(f"{i:02d}: {col}")

# Normalize column names for robust matching, while retaining original names.
column_lookup = {str(c).strip().lower(): c for c in df_raw.columns}

timestamp_candidates = [c for c in df_raw.columns if str(c).strip().lower() in {"timestamp", "time", "datetime", "date"}]
index_candidates = [c for c in df_raw.columns if str(c).strip().lower() in {"index", "id"}]

if not timestamp_candidates:
    raise ValueError("Could not find a timestamp column. Inspect df_raw.columns and set TIMESTAMP_COL manually.")

TIMESTAMP_COL = timestamp_candidates[0]
INDEX_COL = index_candidates[0] if index_candidates else None

print("Timestamp column:", TIMESTAMP_COL)
print("Index column:", INDEX_COL)

## 5. Parse timestamps, sort records and audit sampling cadence

In [ ]:
df = df_raw.copy()

df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors='coerce')
invalid_time_count = df[TIMESTAMP_COL].isna().sum()
print("Rows with invalid timestamps:", invalid_time_count)

df = df.dropna(subset=[TIMESTAMP_COL]).sort_values(TIMESTAMP_COL).reset_index(drop=True)

# Duplicate timestamp audit. Do not silently aggregate duplicates at this stage.
duplicate_timestamp_count = df[TIMESTAMP_COL].duplicated().sum()
print("Duplicate timestamps:", duplicate_timestamp_count)

time_deltas = df[TIMESTAMP_COL].diff().dropna()
print("\\nMost common timestamp differences:")
display(time_deltas.value_counts().head(15).to_frame("count"))

print("\\nTimestamp delta summary:")
display(time_deltas.describe())

print("Start:", df[TIMESTAMP_COL].min())
print("End:", df[TIMESTAMP_COL].max())

### Optional cadence visualization

In [ ]:
delta_seconds = time_deltas.dt.total_seconds()
plt.figure(figsize=(10, 4))
sns.histplot(delta_seconds.clip(upper=delta_seconds.quantile(0.99)), bins=50)
plt.xlabel("Timestamp difference (seconds; clipped at 99th percentile)")
plt.title("MetroPT-3 observed sampling intervals")
plt.tight_layout()
plt.show()

**Interpretation checkpoint:** record the most common interval and the amount of irregularity in your project diary. Do not state “1 Hz” or “one reading every 10 seconds” until this cell confirms it for the downloaded file. This notebook initially uses **60 observations per window**, not a fixed number of minutes.

## 6. Missing values, duplicates and sensor columns

In [ ]:
print("Missing values by column:")
missing_report = df.isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing_report["missing_pct"] = 100 * missing_report["missing_count"] / len(df)
display(missing_report)

print("Exact duplicate rows:", df.duplicated().sum())

# Sensor columns = all columns other than timestamp and a possible row index.
exclude_cols = [TIMESTAMP_COL]
if INDEX_COL is not None:
    exclude_cols.append(INDEX_COL)

SENSOR_COLS = [c for c in df.columns if c not in exclude_cols]
print("Candidate sensor columns:", SENSOR_COLS)
print("Number of candidate sensor columns:", len(SENSOR_COLS))

# Convert sensor values to numeric where possible.
for col in SENSOR_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

sensor_numeric_cols = [c for c in SENSOR_COLS if pd.api.types.is_numeric_dtype(df[c])]
print("Numeric sensor columns:", sensor_numeric_cols)

# Identify binary / low-cardinality columns from observed values, not from assumptions.
sensor_type_rows = []
for col in sensor_numeric_cols:
    nunique = df[col].nunique(dropna=True)
    vals = set(df[col].dropna().unique()[:20])
    is_binary = nunique <= 2
    sensor_type_rows.append({
        "column": col,
        "unique_values": nunique,
        "min": df[col].min(),
        "max": df[col].max(),
        "likely_binary": is_binary
    })
sensor_type_report = pd.DataFrame(sensor_type_rows)
display(sensor_type_report)

## 7. Basic descriptive statistics

In [ ]:
display(df[sensor_numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

## 8. Visualize selected sensor signals

Plot a manageable sample rather than sending all 1.5 million points to the renderer.

In [ ]:
plot_df = df.set_index(TIMESTAMP_COL)
plot_step = max(1, len(plot_df) // 20000)
plot_sample = plot_df.iloc[::plot_step]

selected_for_plot = sensor_numeric_cols[:8]
fig, axes = plt.subplots(len(selected_for_plot), 1, figsize=(15, 2.4 * len(selected_for_plot)), sharex=True)
if len(selected_for_plot) == 1:
    axes = [axes]

for ax, col in zip(axes, selected_for_plot):
    ax.plot(plot_sample.index, plot_sample[col], linewidth=0.6)
    ax.set_ylabel(col)
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Timestamp")
fig.suptitle("MetroPT-3 sensor signals (downsampled for visualization)", y=1.002)
plt.tight_layout()
plt.show()

## 9. Documented failure intervals

The UCI dataset description reports four air-leak failure periods. These are weak event annotations, not dense per-row fault labels. Verify these dates against the dataset paper/repository documentation before final submission. Times below are interpreted as local dataset timestamps; if the source file uses a different timezone convention, document and adjust it.

In [ ]:
FAILURE_EVENTS = [
    {
        "event_id": "air_leak_1",
        "start": "2020-04-18 00:00:00",
        "end":   "2020-04-18 23:59:00",
        "description": "Reported air-leak failure"
    },
    {
        "event_id": "air_leak_2",
        "start": "2020-05-29 23:30:00",
        "end":   "2020-05-30 06:00:00",
        "description": "Reported air-leak failure"
    },
    {
        "event_id": "air_leak_3",
        "start": "2020-06-05 10:00:00",
        "end":   "2020-06-07 14:30:00",
        "description": "Reported air-leak failure"
    },
    {
        "event_id": "air_leak_4",
        "start": "2020-07-15 14:30:00",
        "end":   "2020-07-15 19:00:00",
        "description": "Reported air-leak failure"
    },
]

events_df = pd.DataFrame(FAILURE_EVENTS)
events_df["start"] = pd.to_datetime(events_df["start"])
events_df["end"] = pd.to_datetime(events_df["end"])
display(events_df)

## 10. Attach event annotations to timestamps

`in_failure_interval` marks timestamps inside documented event intervals. It is useful for evaluation and visualization, but it does not mean every sensor row has a verified fault label.

In [ ]:
df["in_failure_interval"] = False
for event in FAILURE_EVENTS:
    start = pd.Timestamp(event["start"])
    end = pd.Timestamp(event["end"])
    mask = df[TIMESTAMP_COL].between(start, end, inclusive="both")
    df.loc[mask, "in_failure_interval"] = True

print("Rows inside documented failure intervals:", int(df["in_failure_interval"].sum()))
print("Fraction of rows inside documented intervals:", df["in_failure_interval"].mean())

# Plot a selected sensor with failure intervals shaded.
sensor_to_inspect = sensor_numeric_cols[0]
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(plot_sample.index, plot_sample[sensor_to_inspect], linewidth=0.6, label=sensor_to_inspect)
for event in FAILURE_EVENTS:
    ax.axvspan(pd.Timestamp(event["start"]), pd.Timestamp(event["end"]), alpha=0.18)
ax.set_title(f"{sensor_to_inspect} with reported air-leak intervals shaded")
ax.set_xlabel("Timestamp")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Chronological split

Initial anomaly-detection protocol:
- Train: February 2020
- Validation: March 2020
- Test: April through August 2020

This follows a time-ordered evaluation principle. The actual dataset date coverage is checked below. If your downloaded version has different coverage, adjust the boundaries and explain why.

In [ ]:
TRAIN_START = pd.Timestamp("2020-02-01")
TRAIN_END   = pd.Timestamp("2020-03-01")  # exclusive
VAL_START   = pd.Timestamp("2020-03-01")
VAL_END     = pd.Timestamp("2020-04-01")  # exclusive
TEST_START  = pd.Timestamp("2020-04-01")
TEST_END    = pd.Timestamp("2020-09-01")  # exclusive

train_df = df[(df[TIMESTAMP_COL] >= TRAIN_START) & (df[TIMESTAMP_COL] < TRAIN_END)].copy()
val_df   = df[(df[TIMESTAMP_COL] >= VAL_START) & (df[TIMESTAMP_COL] < VAL_END)].copy()
test_df  = df[(df[TIMESTAMP_COL] >= TEST_START) & (df[TIMESTAMP_COL] < TEST_END)].copy()

split_summary = pd.DataFrame([
    {"split": "train", "rows": len(train_df), "start": train_df[TIMESTAMP_COL].min(), "end": train_df[TIMESTAMP_COL].max()},
    {"split": "validation", "rows": len(val_df), "start": val_df[TIMESTAMP_COL].min(), "end": val_df[TIMESTAMP_COL].max()},
    {"split": "test", "rows": len(test_df), "start": test_df[TIMESTAMP_COL].min(), "end": test_df[TIMESTAMP_COL].max()},
])
display(split_summary)

if min(len(train_df), len(val_df), len(test_df)) == 0:
    raise ValueError("One or more chronological splits are empty. Inspect date coverage and update split boundaries.")

print("Failure rows by split:")
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(name, int(part["in_failure_interval"].sum()))

**Critical limitation:** the first-month training split is appropriate for learning normal sensor behaviour, but it does not provide positive failure examples for supervised failure classification. The four reported events are too few to support strong claims from a conventional random train/test split. We will therefore make unsupervised anomaly detection the main Phase 2 model and treat future-failure classification as exploratory.

## 12. Handle missing values without leakage

Fit imputation values on the training period only. The same training medians are applied to validation and test. We do not interpolate across the full dataset because that could use future values.

In [ ]:
# Keep only numeric sensor columns.
FEATURE_COLS = sensor_numeric_cols.copy()

train_medians = train_df[FEATURE_COLS].median()
if train_medians.isna().any():
    all_nan = train_medians[train_medians.isna()].index.tolist()
    raise ValueError(f"These features are entirely missing in the training split: {all_nan}")

def apply_train_medians(part):
    out = part.copy()
    out[FEATURE_COLS] = out[FEATURE_COLS].fillna(train_medians)
    return out

train_df = apply_train_medians(train_df)
val_df = apply_train_medians(val_df)
test_df = apply_train_medians(test_df)

print("Remaining missing values in train:", int(train_df[FEATURE_COLS].isna().sum().sum()))
print("Remaining missing values in validation:", int(val_df[FEATURE_COLS].isna().sum().sum()))
print("Remaining missing values in test:", int(test_df[FEATURE_COLS].isna().sum()))

## 13. Scale features using training data only

In [ ]:
scaler = StandardScaler()
X_train_rows = scaler.fit_transform(train_df[FEATURE_COLS]).astype(np.float32)
X_val_rows   = scaler.transform(val_df[FEATURE_COLS]).astype(np.float32)
X_test_rows  = scaler.transform(test_df[FEATURE_COLS]).astype(np.float32)

print("Train matrix:", X_train_rows.shape)
print("Validation matrix:", X_val_rows.shape)
print("Test matrix:", X_test_rows.shape)

joblib.dump(scaler, ARTIFACT_DIR / "metropt3_standard_scaler.joblib")
with open(ARTIFACT_DIR / "feature_columns.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)

## 14. Check timestamp gaps inside each split

Sequence windows should not cross large gaps. We define a gap using a configurable multiple of the median positive interval. Inspect the output and adjust if the dataset's cadence is highly irregular.

In [ ]:
def get_gap_threshold_seconds(part, multiplier=5.0):
    deltas = part[TIMESTAMP_COL].sort_values().diff().dt.total_seconds().dropna()
    positive = deltas[deltas > 0]
    if len(positive) == 0:
        raise ValueError("No positive timestamp intervals found.")
    median_delta = float(positive.median())
    return median_delta, median_delta * multiplier

for split_name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    med, threshold = get_gap_threshold_seconds(part)
    gaps = part[TIMESTAMP_COL].sort_values().diff().dt.total_seconds()
    print(f"{split_name}: median interval={med:.3f}s, gap threshold={threshold:.3f}s, large gaps={(gaps > threshold).sum()}")

## 15. Build sliding windows without crossing large gaps

`WINDOW_SIZE=60` means 60 consecutive observations. The window builder resets at timestamp gaps and returns the timestamp of the final observation in each window. No window crosses a train/validation/test boundary.

In [ ]:
WINDOW_SIZE = 60
GAP_MULTIPLIER = 5.0

def build_windows(part, X_rows, window_size=WINDOW_SIZE, gap_multiplier=GAP_MULTIPLIER):
    part = part.reset_index(drop=True)
    times = part[TIMESTAMP_COL].reset_index(drop=True)
    y_event = part["in_failure_interval"].astype(np.int8).to_numpy()

    deltas = times.diff().dt.total_seconds().to_numpy()
    positive = deltas[np.isfinite(deltas) & (deltas > 0)]
    median_delta = np.median(positive)
    gap_threshold = median_delta * gap_multiplier

    windows, end_times, end_labels, end_positions = [], [], [], []
    segment_start = 0

    # Split into contiguous segments at large gaps or non-positive deltas.
    boundaries = []
    for i in range(1, len(times)):
        d = deltas[i]
        if (not np.isfinite(d)) or (d <= 0) or (d > gap_threshold):
            boundaries.append(i)
    segment_starts = [0] + boundaries
    segment_ends = boundaries + [len(times)]

    for s, e in zip(segment_starts, segment_ends):
        if e - s < window_size:
            continue
        for end in range(s + window_size - 1, e):
            start = end - window_size + 1
            windows.append(X_rows[start:end + 1])
            end_times.append(times.iloc[end])
            end_labels.append(y_event[end])
            end_positions.append(end)

    if not windows:
        return (
            np.empty((0, window_size, X_rows.shape[1]), dtype=np.float32),
            pd.to_datetime([]),
            np.empty((0,), dtype=np.int8),
            np.empty((0,), dtype=np.int64)
        )

    return (
        np.stack(windows).astype(np.float32),
        pd.to_datetime(end_times),
        np.asarray(end_labels, dtype=np.int8),
        np.asarray(end_positions, dtype=np.int64)
    )

X_train, train_window_times, y_train_event, train_end_pos = build_windows(train_df, X_train_rows)
X_val, val_window_times, y_val_event, val_end_pos = build_windows(val_df, X_val_rows)
X_test, test_window_times, y_test_event, test_end_pos = build_windows(test_df, X_test_rows)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("Failure-window counts (end timestamp inside event):",
      int(y_train_event.sum()), int(y_val_event.sum()), int(y_test_event.sum()))

## 16. Prepare normal-only training windows for an LSTM Autoencoder

The anomaly detector learns reconstruction patterns from the training period. Because the training period is intended to represent normal operation, exclude any window whose final timestamp is inside a documented event. For stricter filtering, you can exclude any window overlapping an event interval.

In [ ]:
# Training split should be checked for documented events before training.
if y_train_event.sum() > 0:
    print("WARNING: Training period contains documented failure windows. Excluding them.")
    X_train_normal = X_train[y_train_event == 0]
else:
    X_train_normal = X_train

print("Normal training windows:", X_train_normal.shape)
if len(X_train_normal) == 0:
    raise ValueError("No normal training windows available. Revisit the date split or event annotations.")

## 17. Create future-failure warning labels (exploratory)

For a timestamp `t`, the target is 1 if a documented failure **starts** in `(t, t + horizon]`. Rows already inside a failure interval are excluded from this future-onset task. The final horizon-sized portion of the dataset is censored because its future is not observable.

Because there are only four documented events, this task is highly event-scarce. Do not randomly split overlapping windows. Use event-level or chronological evaluation and report uncertainty.

In [ ]:
def make_future_failure_labels(times, events, horizon_hours=6):
    times = pd.Series(pd.to_datetime(times)).reset_index(drop=True)
    horizon = pd.Timedelta(hours=horizon_hours)
    labels = np.zeros(len(times), dtype=np.int8)
    valid = np.ones(len(times), dtype=bool)

    event_starts = sorted(pd.Timestamp(e["start"]) for e in events)

    for i, t in enumerate(times):
        future_end = t + horizon
        # A label is valid only if the full prediction horizon is observable
        # within the available timeline.
        if future_end > times.iloc[-1]:
            valid[i] = False
            continue
        # Do not label a timestamp during an already ongoing reported event.
        currently_in_event = any(pd.Timestamp(e["start"]) <= t <= pd.Timestamp(e["end"]) for e in events)
        if currently_in_event:
            valid[i] = False
            continue
        labels[i] = int(any(t < start <= future_end for start in event_starts))

    return labels, valid

# Create labels for row timestamps in each split.
HORIZON_HOURS = 6
y_train_future, train_future_valid = make_future_failure_labels(train_df[TIMESTAMP_COL], FAILURE_EVENTS, HORIZON_HOURS)
y_val_future, val_future_valid = make_future_failure_labels(val_df[TIMESTAMP_COL], FAILURE_EVENTS, HORIZON_HOURS)
y_test_future, test_future_valid = make_future_failure_labels(test_df[TIMESTAMP_COL], FAILURE_EVENTS, HORIZON_HOURS)

print("Future-failure positive rows:")
print("Train:", int(y_train_future[train_future_valid].sum()), "/", int(train_future_valid.sum()))
print("Validation:", int(y_val_future[val_future_valid].sum()), "/", int(val_future_valid.sum()))
print("Test:", int(y_test_future[test_future_valid].sum()), "/", int(test_future_valid.sum()))

### Map future-failure labels to the end of each sequence window

In [ ]:
def map_row_labels_to_windows(X_windows, window_end_positions, row_labels, row_valid):
    labels = row_labels[window_end_positions]
    valid = row_valid[window_end_positions]
    return X_windows[valid], labels[valid], valid

X_train_future, y_train_future_windows, train_future_window_valid = map_row_labels_to_windows(
    X_train, train_end_pos, y_train_future, train_future_valid
)
X_val_future, y_val_future_windows, val_future_window_valid = map_row_labels_to_windows(
    X_val, val_end_pos, y_val_future, val_future_valid
)
X_test_future, y_test_future_windows, test_future_window_valid = map_row_labels_to_windows(
    X_test, test_end_pos, y_test_future, test_future_valid
)

print("Valid future-warning windows:")
for name, X_part, y_part in [
    ("Train", X_train_future, y_train_future_windows),
    ("Validation", X_val_future, y_val_future_windows),
    ("Test", X_test_future, y_test_future_windows),
]:
    print(name, "windows:", len(X_part), "positives:", int(y_part.sum()) if len(y_part) else 0)


## 18. Baseline anomaly threshold from validation data

For Phase 1 we only prepare a threshold baseline. Phase 2 will train an LSTM Autoencoder and compute reconstruction errors. The threshold must be selected on validation data, not tuned on the test set.

In [ ]:
# Placeholder baseline: distance from the training-normal mean in scaled feature space.
# This is NOT the final anomaly model. It is a simple sanity-check score.
train_center = X_train_rows.mean(axis=0)
train_scale = X_train_rows.std(axis=0) + 1e-6

def simple_window_score(X):
    # Average absolute standardized deviation across time and features.
    z = (X - train_center.reshape(1, 1, -1)) / train_scale.reshape(1, 1, -1)
    return np.mean(np.abs(z), axis=(1, 2))

val_scores = simple_window_score(X_val)
test_scores = simple_window_score(X_test)

# Use the 99th percentile of validation scores as a preliminary threshold.
# This is a baseline threshold only; later compare 95%, 99%, 99.5%, etc.
anomaly_threshold_baseline = float(np.quantile(val_scores, 0.99))
print("Baseline validation threshold (99th percentile):", anomaly_threshold_baseline)

y_test_anomaly_pred_baseline = (test_scores > anomaly_threshold_baseline).astype(np.int8)

## 19. Evaluate baseline anomaly flags carefully

Point-level precision/recall is only a first diagnostic. Because failure labels are sparse intervals and timestamps are autocorrelated, also report event-level detection, false alarms per day and warning lead time in the final research paper.

In [ ]:
if y_test_event.sum() > 0:
    p, r, f1, _ = precision_recall_fscore_support(
        y_test_event, y_test_anomaly_pred_baseline, average="binary", zero_division=0
    )
    print({"precision": p, "recall": r, "f1": f1})
    print(classification_report(y_test_event, y_test_anomaly_pred_baseline, zero_division=0))
    print("Confusion matrix:\\n", confusion_matrix(y_test_event, y_test_anomaly_pred_baseline))
else:
    print("No documented failure rows in the test split; cannot compute point-level event metrics.")

## 20. Visualize test anomaly scores and reported events

In [ ]:
score_plot = pd.DataFrame({
    "timestamp": test_window_times,
    "score": test_scores,
    "reported_failure_at_window_end": y_test_event
})

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(score_plot["timestamp"], score_plot["score"], linewidth=0.7, label="Baseline anomaly score")
ax.axhline(anomaly_threshold_baseline, linestyle="--", label="Validation 99th percentile threshold")
for event in FAILURE_EVENTS:
    ax.axvspan(pd.Timestamp(event["start"]), pd.Timestamp(event["end"]), alpha=0.18)
ax.set_title("Test anomaly score with reported failure intervals")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Anomaly score")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 21. Robustness experiment: inject missingness and noise

The source dataset may have few or no missing values, but a deployed monitoring system can experience sensor dropouts and noise. This controlled experiment creates perturbed copies for robustness testing. Keep the original untouched and clearly label these as synthetic perturbations.

In [ ]:
def inject_sensor_corruption(X, missing_rate=0.02, noise_std=0.05, seed=42):
    rng = np.random.default_rng(seed)
    X_corrupt = X.copy()
    mask = rng.random(X_corrupt.shape) < missing_rate
    X_corrupt[mask] = 0.0  # zero is the scaled-space neutral value
    X_corrupt += rng.normal(0, noise_std, size=X_corrupt.shape).astype(np.float32)
    return X_corrupt, mask

if len(X_test) > 0:
    X_test_corrupt, corruption_mask = inject_sensor_corruption(X_test[:min(1000, len(X_test))])
    print("Original subset shape:", X_test[:min(1000, len(X_test))].shape)
    print("Corrupted subset shape:", X_test_corrupt.shape)
    print("Injected missing-value fraction:", corruption_mask.mean())
else:
    print("No test windows available for corruption experiment.")

## 22. Save processed arrays, event table and metadata

In [ ]:
# Save compressed arrays. Use separate arrays for each chronological split.
np.savez_compressed(
    PROCESSED_DIR / "metropt3_phase1_windows.npz",
    X_train=X_train,
    X_train_normal=X_train_normal,
    X_val=X_val,
    X_test=X_test,
    X_train_future=X_train_future,
    X_val_future=X_val_future,
    X_test_future=X_test_future,
    train_window_times=train_window_times.astype("datetime64[ns]").astype("int64"),
    val_window_times=val_window_times.astype("datetime64[ns]").astype("int64"),
    test_window_times=test_window_times.astype("datetime64[ns]").astype("int64"),
    y_train_event=y_train_event,
    y_val_event=y_val_event,
    y_test_event=y_test_event,
    y_train_future=y_train_future_windows,
    y_val_future=y_val_future_windows,
    y_test_future=y_test_future_windows,
)

events_df.to_csv(PROCESSED_DIR / "metropt3_failure_events.csv", index=False)

metadata = {
    "dataset": "MetroPT-3",
    "uci_dataset_id": 791,
    "dataset_doi": "10.24432/C5VW3R",
    "timestamp_column": str(TIMESTAMP_COL),
    "index_column": str(INDEX_COL) if INDEX_COL is not None else None,
    "feature_columns": [str(c) for c in FEATURE_COLS],
    "window_size_observations": int(WINDOW_SIZE),
    "gap_multiplier": float(GAP_MULTIPLIER),
    "train_period": [str(TRAIN_START), str(TRAIN_END)],
    "validation_period": [str(VAL_START), str(VAL_END)],
    "test_period": [str(TEST_START), str(TEST_END)],
    "future_failure_horizon_hours": int(HORIZON_HOURS),
    "failure_event_count": len(FAILURE_EVENTS),
    "important_caveat": "MetroPT-3 has sparse reported failure intervals and no continuous supervised RUL labels."
}
with open(ARTIFACT_DIR / "phase1_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved files:")
for path in [PROCESSED_DIR / "metropt3_phase1_windows.npz",
             PROCESSED_DIR / "metropt3_failure_events.csv",
             ARTIFACT_DIR / "metropt3_standard_scaler.joblib",
             ARTIFACT_DIR / "feature_columns.json",
             ARTIFACT_DIR / "phase1_metadata.json"]:
    print(path, "exists:", path.exists(), "size_MB:", round(path.stat().st_size / (1024**2), 2) if path.exists() else None)

## 23. Final Phase 1 checklist

Before moving to Phase 2, confirm:

- [ ] The dataset downloaded from the official UCI source.
- [ ] You inspected the actual column names and number of sensor fields.
- [ ] You recorded the observed sampling cadence and timestamp irregularities.
- [ ] You checked duplicate timestamps, missing values and sensor dtypes.
- [ ] You verified the four event annotations against the UCI documentation/paper.
- [ ] Train, validation and test are chronological and non-overlapping.
- [ ] The scaler and imputation values were fitted only on training data.
- [ ] Sliding windows do not cross split boundaries or large timestamp gaps.
- [ ] You understand that 60 observations is not necessarily 60 seconds.
- [ ] You have not claimed continuous RUL ground truth or strong supervised results from four events.

### What comes next — Phase 2
1. Train an LSTM Autoencoder on `X_train_normal`.
2. Calibrate its reconstruction-error threshold using `X_val`.
3. Compare with Isolation Forest and a Transformer-based reconstruction/anomaly baseline.
4. Build the hybrid LSTM–Transformer model for exploratory future-failure warning, using event-aware evaluation.
5. Add uncertainty estimation and feature attribution.
6. Connect model outputs to FastAPI and the React dashboard.

**Expected Phase 1 outputs:** `metropt3_phase1_windows.npz`, scaler, feature list, event CSV and metadata JSON under your Google Drive project folder.